# Partie III — Réseaux Récurrents : RNN, LSTM, GRU et Seq2Seq

Les séquences posent un défi fondamental en apprentissage automatique : **l'ordre compte**, et les longueurs peuvent varier d'un exemple à l'autre. Un réseau de neurones classique (MLP) traite chaque entrée de manière indépendante et ne sait pas exploiter l'information temporelle ou contextuelle.

Les **réseaux récurrents** répondent à ce défi en maintenant un **état caché** qui se propage d'un pas de temps au suivant, permettant au modèle de "se souvenir" des tokens précédents.

**Applications couvertes dans ce notebook :**
- **Classification de texte** (analyse de sentiment) avec RNN, LSTM et GRU
- **Traduction automatique jouet** (nombres → mots) avec une architecture Seq2Seq
- **Génération de séquences** : décodage glouton et beam search

D'autres applications importantes incluent la génération de texte, la reconnaissance vocale, la prédiction de séries temporelles, et le résumé automatique.

## 1. Théorie : Modèles de Langage et Séquences

### Probabilité d'une séquence

Un modèle de langage estime la probabilité d'une séquence de tokens $(x_1, x_2, \ldots, x_n)$. Par la **règle de la chaîne** des probabilités :

$$P(x_1, x_2, \ldots, x_n) = \prod_{t=1}^{n} P(x_t \mid x_1, \ldots, x_{t-1})$$

Chaque token est prédit conditionnellement à tout le contexte précédent. Le rôle du réseau récurrent est d'approximer cette distribution conditionnelle.

---

### RNN (Recurrent Neural Network)

Le RNN maintient un **état caché** $h_t$ mis à jour à chaque pas de temps :

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b)$$

Le même jeu de paramètres $(W_{xh}, W_{hh}, b)$ est appliqué à **chaque pas de temps** — c'est le **partage de poids temporel** (*weight tying*). Cela permet au modèle de traiter des séquences de longueur variable avec un nombre fixe de paramètres.

**Problème — gradients qui disparaissent/explosent (BPTT) :**  
Lors de la rétropropagation à travers le temps (*Backpropagation Through Time*), le gradient de la perte par rapport à un état caché lointain implique un produit de matrices de transition :

$$\frac{\partial h_t}{\partial h_0} = \prod_{k=1}^{t} \frac{\partial h_k}{\partial h_{k-1}}$$

Sur des séquences longues, ce produit s'**éteint** (valeurs propres $< 1$) ou **explose** (valeurs propres $> 1$), rendant l'apprentissage de dépendances à long terme impossible.

---

### LSTM (Long Short-Term Memory)

Le LSTM résout le problème du gradient disparaissant grâce à une **cellule mémoire** $c_t$ contrôlée par des **portes** :

| Porte | Formule | Rôle |
|-------|---------|------|
| Oubli | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | Que garder de la mémoire précédente ? |
| Entrée | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | Quelle nouvelle info intégrer ? |
| Candidat | $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$ | Nouvelle information candidate |
| Cellule | $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ | Mise à jour de la mémoire |
| Sortie | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$, $\;h_t = o_t \odot \tanh(c_t)$ | État caché exposé |

Le chemin additif $c_t = f_t \odot c_{t-1} + \ldots$ permet aux gradients de **circuler sans atténuation** sur de longues distances.

---

### GRU (Gated Recurrent Unit)

Le GRU est une version **simplifiée** du LSTM avec seulement 2 portes :
- **Porte de réinitialisation** (*reset gate*) : contrôle combien de l'état passé est utilisé pour calculer le nouvel état candidat.
- **Porte de mise à jour** (*update gate*) : interpolation entre l'état passé et le nouvel état candidat.

Le GRU a **moins de paramètres** que le LSTM (pas de cellule mémoire séparée) et converge souvent aussi vite sur des tâches de taille moyenne.

---

### Perplexité

La **perplexité** mesure la « surprise » moyenne du modèle face aux données de validation :

$$\text{PPL} = \exp\!\left(\frac{1}{N}\sum_{i=1}^{N} H(y_i, \hat{y}_i)\right)$$

où $H$ est l'entropie croisée. Interprétation intuitive : une perplexité de $k$ signifie que le modèle hésite en moyenne entre $k$ tokens également probables.
- **PPL = 1** : le modèle est parfait (certitude totale).
- **PPL = taille du vocabulaire** : le modèle ne fait pas mieux que le hasard.
- **PPL faible** = bon modèle de langage.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

from deep_learning_project.utils import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Device : {device}")

## 2. Partie A — Classification de Sentiment

### Données et Pipeline Textuel

Nous utilisons un **jeu de données synthétique de sentiment** composé de :
- **1 000 phrases positives** : construites à partir de modèles avec des adjectifs comme *excellent*, *magnifique*, *parfait*, etc.
- **1 000 phrases négatives** : construites avec des termes comme *horrible*, *décevant*, *inutile*, etc.

Le pipeline textuel comprend :
1. **Tokenisation** : découpage des phrases en tokens (mots).
2. **Construction du vocabulaire** : on conserve uniquement les tokens apparaissant au moins `min_vocab_freq` fois ; les autres sont remplacés par `<UNK>`.
3. **Tokens spéciaux** : `<PAD>` (rembourrage), `<UNK>` (inconnu), `<SOS>` (début de séquence), `<EOS>` (fin de séquence).
4. **Padding** : toutes les séquences d'un même batch sont ramenées à la même longueur (longueur maximale du batch) en ajoutant des tokens `<PAD>` à droite.
5. **Pack/Unpack** : pour éviter que le RNN traite les tokens de padding, on utilise `pack_padded_sequence` de PyTorch, qui ignore les positions paddées lors du calcul.

In [ ]:
from deep_learning_project.data.text import (
    get_sentiment_loaders, Vocabulary, make_sentiment_dataset
)

# Charger les données
train_loader, val_loader, vocab = get_sentiment_loaders(
    n_samples=2000, val_size=0.2, batch_size=32, seed=42, min_vocab_freq=2
)

print(f"Taille du vocabulaire : {len(vocab)} tokens")
print(f"Tokens spéciaux : PAD={vocab.token2idx['<PAD>']}, UNK={vocab.token2idx['<UNK>']}, "
      f"SOS={vocab.token2idx['<SOS>']}, EOS={vocab.token2idx['<EOS>']}")

# Afficher quelques exemples
texts, labels = make_sentiment_dataset(n_samples=2000, seed=42)
print(f"\nExemples de données :")
for i in [0, 1, 1000, 1001]:
    encoded = vocab.encode(texts[i])
    sentiment = "Positif" if labels[i] == 1 else "Négatif"
    print(f"  [{sentiment}] '{texts[i][:60]}...' -> {encoded[:8]}...")

# Distribution
unique, counts = np.unique(labels, return_counts=True)
print(f"\nDistribution : {dict(zip(['Négatif', 'Positif'], counts))}")

## 3. Entraînement : RNN, LSTM, GRU

Nous entraînons les trois architectures avec **exactement les mêmes hyperparamètres** pour une comparaison équitable :

| Hyperparamètre | Valeur |
|----------------|--------|
| Dimension d'embedding | 64 |
| Dimension cachée | 128 |
| Nombre de couches | 1 |
| Dropout | 0.3 |
| Optimiseur | Adam (lr=1e-3) |
| Époques | 20 |

Architecture commune : **Embedding → RNN/LSTM/GRU → Dernier état caché → Couche linéaire de classification**.

Le dernier état caché (à la position correspondant à la vraie longueur de la séquence, avant le padding) sert de représentation de la phrase entière.

In [ ]:
from deep_learning_project.models.rnn import VanillaRNN, LSTMModel, GRUModel
from deep_learning_project.training.train_mlp import train

VOCAB_SIZE = len(vocab)
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 1
NUM_CLASSES = 2
DROPOUT = 0.3
N_EPOCHS = 20

criterion = nn.CrossEntropyLoss()
rnn_histories = {}
rnn_models = {}

model_classes = {
    "VanillaRNN": VanillaRNN,
    "LSTM":       LSTMModel,
    "GRU":        GRUModel,
}

for name, ModelClass in model_classes.items():
    set_seed(42)
    model = ModelClass(
        vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS, num_classes=NUM_CLASSES, dropout_p=DROPOUT,
        bidirectional=False,
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = train(
        model, train_loader, val_loader, optimizer, criterion,
        device, epochs=N_EPOCHS,
        checkpoint_filename=f"rnn_{name.lower()}_best.pt",
        verbose=False,
    )
    rnn_histories[name] = history
    rnn_models[name] = model
    print(f"[{name:10s}] best_val_acc={max(history['val_acc']):.4f} "
          f"best_val_loss={min(history['val_loss']):.4f}")

## 4. Problèmes d'Entraînement : Gradients Explosifs

Les RNNs à plusieurs couches entraînés avec SGD et un taux d'apprentissage élevé sont particulièrement **sujets aux gradients explosifs**. Sans contre-mesure, les gradients peuvent atteindre des normes très grandes, ce qui rend l'optimisation instable (les paramètres font des sauts énormes).

La solution standard est le **gradient clipping** (`torch.nn.utils.clip_grad_norm_`) : si la norme L2 globale des gradients dépasse un seuil $\tau$, on rescale tous les gradients par le facteur $\tau / \|\nabla\|$ :

$$g \leftarrow g \cdot \frac{\tau}{\max(\|g\|_2,\, \tau)}$$

Cette opération ne change pas la **direction** du gradient, seulement sa **magnitude**. Elle est appliquée **après** `loss.backward()` et **avant** `optimizer.step()`.

Nous allons démontrer ce phénomène ci-dessous avec un VanillaRNN à 2 couches et SGD (combinaison délibérément instable).

In [ ]:
# Démonstration : norme des gradients avec et sans clipping
set_seed(42)
demo_rnn = VanillaRNN(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=2,  # Plus de couches = plus de risque d'explosion
    num_classes=NUM_CLASSES, dropout_p=0.0,
).to(device)
optimizer_demo = torch.optim.SGD(demo_rnn.parameters(), lr=0.1)  # SGD + lr élevé = plus instable

grad_norms_no_clip = []
grad_norms_with_clip = []
CLIP_VALUE = 1.0

demo_rnn.train()
for batch_idx, (seqs, lengths, labels) in enumerate(train_loader):
    if batch_idx >= 30:
        break
    seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
    optimizer_demo.zero_grad()
    logits = demo_rnn(seqs, lengths)
    loss = criterion(logits, labels)
    loss.backward()
    
    # Norme des gradients AVANT clipping
    total_norm = 0.0
    for p in demo_rnn.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    total_norm = total_norm ** 0.5
    grad_norms_no_clip.append(total_norm)
    
    # Appliquer le clipping
    clipped_norm = torch.nn.utils.clip_grad_norm_(demo_rnn.parameters(), CLIP_VALUE)
    grad_norms_with_clip.append(min(float(clipped_norm), CLIP_VALUE))
    
    optimizer_demo.step()

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(grad_norms_no_clip, label="Sans clipping", color="red", alpha=0.7)
ax.plot(grad_norms_with_clip, label=f"Avec clipping (max={CLIP_VALUE})", color="green")
ax.axhline(y=CLIP_VALUE, color="orange", linestyle="--", label=f"Seuil = {CLIP_VALUE}")
ax.set_title("Norme des Gradients : Gradients Explosifs")
ax.set_xlabel("Étape d'entraînement")
ax.set_ylabel("Norme L2 des gradients")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
from deep_learning_project.utils.io import save_figure
save_figure(fig, "rnn_gradient_clipping.png")
plt.show()
print(f"Norme max sans clipping : {max(grad_norms_no_clip):.2f}")
print(f"Norme max avec clipping : {max(grad_norms_with_clip):.2f}")

## 5. Comparaison : RNN vs LSTM vs GRU

Comparons les courbes d'entraînement et les métriques finales (accuracy, F1-score, perplexité) pour les trois architectures entraînées précédemment.

In [ ]:
from deep_learning_project.evaluation.classification import (
    get_predictions, compute_accuracy, compute_f1, plot_confusion_matrix, plot_training_history
)
from deep_learning_project.utils.io import load_checkpoint, save_metrics_json

# Courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {"VanillaRNN": "red", "LSTM": "blue", "GRU": "green"}
for name, history in rnn_histories.items():
    axes[0].plot(history["val_loss"], label=name, color=colors[name])
    axes[1].plot(history["val_acc"], label=name, color=colors[name])
axes[0].set_title("Loss validation")
axes[0].set_xlabel("Époque"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("Accuracy validation")
axes[1].set_xlabel("Époque"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, "rnn_comparison_curves.png")
plt.show()

# Métriques finales
print(f"\n{'Modèle':12s} | {'Accuracy':>10s} | {'F1-Score':>10s} | {'Perplexité':>12s}")
print("-" * 55)
comparison_metrics = {}
for name, model in rnn_models.items():
    load_checkpoint(model, f"rnn_{name.lower()}_best.pt", device)
    y_true, y_pred = get_predictions(model, val_loader, device)
    acc = compute_accuracy(y_true, y_pred)
    f1 = compute_f1(y_true, y_pred)
    
    # Perplexité = exp(avg cross-entropy)
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for seqs, lengths, labels in val_loader:
            seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
            logits = model(seqs, lengths)
            total_loss += criterion(logits, labels).item()
            n_batches += 1
    ppl = np.exp(total_loss / n_batches)
    
    comparison_metrics[name] = {"accuracy": acc, "f1": f1, "perplexity": ppl}
    print(f"{name:12s} | {acc:>10.4f} | {f1:>10.4f} | {ppl:>12.2f}")

save_metrics_json(comparison_metrics, "rnn_comparison_metrics.json")

## 6. Théorie : Seq2Seq — Encodeur-Décodeur

### Architecture Encodeur-Décodeur

L'architecture **Seq2Seq** (séquence à séquence) est conçue pour transformer une séquence d'entrée en une séquence de sortie de longueur potentiellement différente. Elle se compose de deux parties :

1. **Encodeur** : lit la séquence source token par token et produit un **vecteur de contexte** $c$ (généralement le dernier état caché $h_T^{\text{enc}}$). Ce vecteur est censé capturer toute l'information sémantique de la source.

2. **Décodeur** : génère la séquence cible **token par token**, en utilisant $c$ comme état caché initial. À chaque pas $t$, il prédit le token suivant $y_t$ conditionnellement à $c$ et aux tokens déjà générés $y_1, \ldots, y_{t-1}$.

$$P(y_1, \ldots, y_m \mid x_1, \ldots, x_n) = \prod_{t=1}^{m} P(y_t \mid y_1, \ldots, y_{t-1},\, c)$$

---

### Teacher Forcing

Pendant l'**entraînement**, le décodeur peut recevoir soit :
- **Sa propre prédiction** $\hat{y}_{t-1}$ au pas précédent (décodage autorégressif).
- **Le vrai token cible** $y_{t-1}$ (teacher forcing).

Le **teacher forcing** accélère considérablement la convergence car il empêche les erreurs de se propager en cascade. Cependant, il crée un **décalage entraînement/inférence** (*exposure bias*) : pendant l'inférence, le modèle voit ses propres prédictions (potentiellement erronées), une situation qu'il n'a jamais rencontrée à l'entraînement.

**Ratio de teacher forcing** : un hyperparamètre contrôlant la proportion de pas où l'on utilise le vrai token.
- `teacher_force_ratio = 1.0` → toujours le vrai token (teacher forcing pur)
- `teacher_force_ratio = 0.0` → toujours la prédiction du modèle
- En pratique : on utilise un ratio élevé en début d'entraînement, puis on le réduit progressivement (**scheduled teacher forcing**).

## 7. Partie B — Traduction : Nombres → Mots

Pour illustrer le mécanisme Seq2Seq de manière concrète, nous utilisons un **dataset jouet** de traduction de nombres en toutes lettres :

- Source : représentation numérique (`"42"`, `"7"`, `"100"`)
- Cible : représentation en mots anglais (`"forty two"`, `"seven"`, `"one hundred"`)

Ce problème est **simple mais non trivial** : il nécessite de mémoriser les correspondances (unités, dizaines, centaines) et de générer plusieurs tokens dans le bon ordre. Il démontre parfaitement la capacité du Seq2Seq à mapper une séquence courte vers une séquence de longueur variable.

In [ ]:
from deep_learning_project.data.text import (
    get_seq2seq_loaders, make_number_translation_dataset
)

# Aperçu du dataset
sources, targets = make_number_translation_dataset()
print("Exemples de paires (source → cible) :")
for i in [0, 5, 12, 21, 42, 75, 99]:
    print(f"  '{sources[i]}' → '{targets[i]}'")

# Charger les loaders
s2s_train, s2s_val, src_vocab, tgt_vocab = get_seq2seq_loaders(
    batch_size=16, val_size=0.2, seed=42
)
print(f"\nVocabulaire source : {len(src_vocab)} tokens")
print(f"Vocabulaire cible  : {len(tgt_vocab)} tokens")
print(f"Tokens cible (aperçu) : {list(tgt_vocab.token2idx.keys())[:15]}")

In [ ]:
from deep_learning_project.models.seq2seq import Encoder, Decoder, Seq2Seq
from deep_learning_project.training.train_mlp import train_seq2seq

# Construire le modèle Seq2Seq
S2S_EMBED = 32
S2S_HIDDEN = 128
S2S_LAYERS = 1

encoder = Encoder(
    src_vocab_size=len(src_vocab), embed_dim=S2S_EMBED,
    hidden_dim=S2S_HIDDEN, num_layers=S2S_LAYERS, dropout_p=0.1,
    pad_idx=src_vocab.token2idx["<PAD>"],
).to(device)

decoder = Decoder(
    tgt_vocab_size=len(tgt_vocab), embed_dim=S2S_EMBED,
    hidden_dim=S2S_HIDDEN, num_layers=S2S_LAYERS, dropout_p=0.1,
    pad_idx=tgt_vocab.token2idx["<PAD>"],
).to(device)

seq2seq = Seq2Seq(
    encoder=encoder, decoder=decoder,
    tgt_vocab_size=len(tgt_vocab),
    sos_idx=tgt_vocab.token2idx["<SOS>"],
    eos_idx=tgt_vocab.token2idx["<EOS>"],
    pad_idx=tgt_vocab.token2idx["<PAD>"],
).to(device)

total_params = sum(p.numel() for p in seq2seq.parameters())
print(f"Paramètres Seq2Seq : {total_params:,}")

# Entraîner
set_seed(42)
optimizer_s2s = torch.optim.Adam(seq2seq.parameters(), lr=1e-3)

s2s_history = train_seq2seq(
    model=seq2seq,
    train_loader=s2s_train,
    val_loader=s2s_val,
    optimizer=optimizer_s2s,
    device=device,
    pad_idx=tgt_vocab.token2idx["<PAD>"],
    epochs=50,
    teacher_force_ratio=0.7,
    grad_clip=1.0,
    checkpoint_filename="seq2seq_best.pt",
    verbose=True,
)

fig = plot_training_history(s2s_history, "Seq2Seq — Courbes d'Entraînement",
                             save_filename="seq2seq_training_curves.png")
plt.show()

## 8. Décodage Glouton (Greedy Decoding)

Le **décodage glouton** est la stratégie la plus simple : à chaque pas de temps, on sélectionne le token avec la **probabilité la plus élevée** :

$$\hat{y}_t = \arg\max_{v \in V} P(y_t = v \mid \hat{y}_1, \ldots, \hat{y}_{t-1}, c)$$

**Avantages :** rapide, coût $O(|V| \cdot T)$ en mémoire et en calcul.

**Inconvénient :** sous-optimal — un choix localement optimal peut mener à une séquence globalement médiocre. Par exemple, si le premier token le plus probable mène à une impasse, le décodage glouton ne peut pas revenir en arrière.

In [ ]:
from deep_learning_project.utils.io import load_checkpoint
load_checkpoint(seq2seq, "seq2seq_best.pt", device)

# Tester sur des exemples
test_pairs = [(sources[i], targets[i]) for i in [0, 7, 13, 21, 42, 55, 77, 99]]
print(f"{'Source':>6} | {'Attendu':>20} | {'Prédit (greedy)':>20} | {'OK?':>4}")
print("-" * 60)

correct = 0
for src_str, tgt_str in test_pairs:
    # Encoder
    src_ids = torch.tensor([src_vocab.encode(src_str)], dtype=torch.long).to(device)
    src_lengths = torch.tensor([src_ids.size(1)], dtype=torch.long)
    
    # Décodage glouton
    decoded_ids = seq2seq.greedy_decode(src_ids, src_lengths, max_len=15)[0]
    decoded_str = tgt_vocab.decode(decoded_ids, skip_special=True)
    
    ok = "✓" if decoded_str.strip() == tgt_str.strip() else "✗"
    if decoded_str.strip() == tgt_str.strip():
        correct += 1
    print(f"{src_str:>6} | {tgt_str:>20} | {decoded_str:>20} | {ok:>4}")

print(f"\nAccuracy glouton : {correct}/{len(test_pairs)} = {correct/len(test_pairs)*100:.0f}%")

## 9. Beam Search

Le **beam search** est une stratégie de décodage qui maintient à chaque pas les **$k$ meilleures hypothèses partielles** (les $k$ séquences les plus probables construites jusqu'à présent).

**Algorithme :**
1. Initialiser avec le token `<SOS>` comme unique hypothèse.
2. À chaque pas, développer chacune des $k$ hypothèses en $|V|$ candidats.
3. Parmi les $k \times |V|$ candidats, ne conserver que les $k$ meilleurs (score = somme des log-probabilités).
4. Répéter jusqu'à ce que toutes les hypothèses se terminent par `<EOS>` ou que la longueur maximale soit atteinte.
5. Retourner l'hypothèse avec le **score normalisé** le plus élevé (division par la longueur pour éviter le biais vers les séquences courtes).

**Comparaison :**
| Méthode | Qualité | Coût |
|---------|---------|------|
| Greedy ($k=1$) | Sous-optimal | $O(|V| \cdot T)$ |
| Beam Search ($k$) | Meilleure | $O(k \cdot |V| \cdot T)$ |
| Recherche exhaustive | Optimal | $O(|V|^T)$ — impossible |

En pratique, $k \in [3, 5]$ offre un bon compromis qualité/vitesse pour la traduction automatique.

In [ ]:
# Implémentation du Beam Search étape par étape
def beam_search_decode(model, src, src_lengths, tgt_vocab, beam_size=3, max_len=15):
    """
    Beam search decoder.
    
    A each step, we maintain beam_size hypotheses.
    Each hypothesis is: (score, token_ids_so_far)
    score = sum of log-probs (lower = better for minimization, we use negative sum)
    """
    model.eval()
    device = src.device
    sos_idx = model.sos_idx
    eos_idx = model.eos_idx
    
    with torch.no_grad():
        # Encoder la source
        hidden, cell = model.encoder(src, src_lengths)
        
        # Initialiser les beams: (log_prob, [token_ids], hidden, cell)
        # Commencer avec <SOS>
        beams = [(0.0, [sos_idx], hidden, cell)]
        completed = []
        
        for step in range(max_len):
            all_candidates = []
            
            for log_prob, tokens, h, c in beams:
                last_token = torch.tensor([tokens[-1]], dtype=torch.long, device=device)
                logits, new_h, new_c = model.decoder(last_token, h, c)
                log_probs = torch.log_softmax(logits, dim=-1).squeeze(0)
                
                # Garder les beam_size meilleurs tokens
                topk_probs, topk_ids = log_probs.topk(beam_size)
                
                for i in range(beam_size):
                    token_id = topk_ids[i].item()
                    new_log_prob = log_prob + topk_probs[i].item()
                    candidate = (new_log_prob, tokens + [token_id], new_h, new_c)
                    
                    if token_id == eos_idx:
                        completed.append((new_log_prob / len(tokens), tokens[1:]))
                    else:
                        all_candidates.append(candidate)
            
            # Trier et garder les beam_size meilleurs
            all_candidates.sort(key=lambda x: x[0], reverse=True)
            beams = all_candidates[:beam_size]
            
            if len(beams) == 0:
                break
        
        # Si aucune séquence complète, prendre le meilleur beam
        if not completed:
            completed = [(beams[0][0] / len(beams[0][1]), beams[0][1][1:])]
        
        # Retourner la meilleure séquence (score le plus élevé)
        completed.sort(key=lambda x: x[0], reverse=True)
        best_ids = completed[0][1]
        return tgt_vocab.decode(best_ids, skip_special=True)


# Comparer greedy vs beam search
print(f"{'Source':>6} | {'Attendu':>20} | {'Greedy':>20} | {'Beam (k=3)':>20}")
print("-" * 75)
greedy_correct = 0
beam_correct = 0
test_extended = [(sources[i], targets[i]) for i in range(0, 100, 10)]

for src_str, tgt_str in test_extended:
    src_ids = torch.tensor([src_vocab.encode(src_str)], dtype=torch.long).to(device)
    src_lengths = torch.tensor([src_ids.size(1)], dtype=torch.long)
    
    # Greedy
    greedy_ids = seq2seq.greedy_decode(src_ids, src_lengths, max_len=15)[0]
    greedy_str = tgt_vocab.decode(greedy_ids, skip_special=True)
    
    # Beam search
    beam_str = beam_search_decode(seq2seq, src_ids, src_lengths, tgt_vocab, beam_size=3)
    
    if greedy_str.strip() == tgt_str.strip(): greedy_correct += 1
    if beam_str.strip() == tgt_str.strip(): beam_correct += 1
    
    print(f"{src_str:>6} | {tgt_str:>20} | {greedy_str:>20} | {beam_str:>20}")

print(f"\nGreedy accuracy : {greedy_correct}/{len(test_extended)} = {greedy_correct/len(test_extended)*100:.0f}%")
print(f"Beam   accuracy : {beam_correct}/{len(test_extended)}  = {beam_correct/len(test_extended)*100:.0f}%")

## 10. Résultats et Interprétation

### RNN vs LSTM vs GRU

- **VanillaRNN** souffre du problème des **gradients disparaissants** sur des séquences de longueur modérée. Il a du mal à apprendre des dépendances à long terme et peut présenter une convergence instable.
- **LSTM** et **GRU** convergent de manière plus stable et plus rapide grâce à leurs mécanismes de portes qui contrôlent le flux d'information.
- **GRU** est plus léger que LSTM (2 portes vs 4 pour LSTM, pas de cellule mémoire séparée) et obtient souvent des performances comparables sur des tâches de classification courtes. Sur des tâches nécessitant une mémoire très longue, le LSTM garde l'avantage.

### Gradient Clipping

La démonstration montre clairement que **sans clipping**, la norme des gradients peut atteindre des valeurs très élevées dès les premières étapes d'entraînement, rendant les mises à jour de poids instables. Le **clipping** (ici avec $\tau = 1.0$) maintient la norme en dessous du seuil sans modifier la direction du gradient, stabilisant l'optimisation.

### Seq2Seq et Teacher Forcing

- Le **teacher forcing** (ratio = 0.7) accélère considérablement l'apprentissage en début d'entraînement car le décodeur reçoit toujours le bon contexte.
- Le risque est l'**exposure bias** : en inférence, le modèle reçoit ses propres prédictions (potentiellement fausses) alors qu'il n'a jamais rencontré cette situation à l'entraînement.
- Le **scheduled teacher forcing** (réduire progressivement le ratio) est une bonne pratique pour atténuer ce problème.

### Beam Search

- Le beam search améliore les résultats en explorant plusieurs hypothèses simultanément, évitant les mauvais choix locaux du décodage glouton.
- $k = 3$ à $k = 5$ est un bon compromis vitesse/qualité pour des vocabulaires cibles de taille modeste.
- La **normalisation par la longueur** est importante pour ne pas pénaliser les séquences longues (la somme des log-probabilités décroît mécaniquement avec la longueur).

## 11. Conclusion

Ce troisième notebook clôt notre exploration des trois grandes familles d'architectures deep learning :

| Architecture | Données adaptées | Inductive bias |
|-------------|-----------------|----------------|
| **MLP** (Notebook 1) | Tableaux structurés, données tabulaires | Aucun biais — apprend toute relation |
| **CNN** (Notebook 2) | Images, signaux spatiaux | Invariance par translation, localité |
| **RNN / LSTM / GRU** (Notebook 3) | Séquences temporelles, texte | Ordre temporel, partage de poids |

**Points clés à retenir :**

1. **RNN** : simple et intuitif, mais souffre des gradients disparaissants/explosifs. Utilisez le **gradient clipping** systématiquement.
2. **LSTM/GRU** : résoudent le problème via des portes apprenables. GRU est plus léger, LSTM plus expressif sur de très longues séquences.
3. **Seq2Seq** : l'architecture encodeur-décodeur est le fondement de la traduction automatique neuronale, du résumé automatique, et de la génération de code.
4. **Teacher forcing** : accélère l'entraînement mais peut créer un décalage entraînement/inférence — à gérer via le scheduled teacher forcing.
5. **Beam search** : améliore la qualité de génération en explorant plusieurs hypothèses, au prix d'un coût computationnel $O(k)$ fois plus élevé.

**Prochaine étape — Transformers :**  
Les architectures récurrentes traitent les séquences **de manière séquentielle**, ce qui limite le parallélisme pendant l'entraînement. Les **Transformers** (Vaswani et al., 2017) résolvent ce problème via le mécanisme d'**attention** : chaque token peut directement « regarder » tous les autres tokens de la séquence en parallèle, sans passer par des états cachés intermédiaires. Cela a révolutionné le traitement du langage naturel et conduit aux modèles comme BERT, GPT, et leurs descendants.